# 02. モデル学習

BioBERTモデルをBC5CDrデータセットでファインチューニングします。

## エンティティ
- **疾患**: 🟠 薄いオレンジ
- **薬剤**: 🔵 薄い青

In [1]:
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from datasets import load_from_disk, DatasetDict
import numpy as np
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.5.1+cu121
GPU: NVIDIA GeForce RTX 3060


In [2]:
# データロード
tokenized_datasets = load_from_disk("processed_bc5cdr")

# ラベル定義（bigbio/bc5cdr: Chemical=薬剤, Disease=疾患）
label_list = ['O', 'B-Chemical', 'I-Chemical', 'B-Disease', 'I-Disease']
num_labels = len(label_list)

id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

print("=== データロード完了 ===")
print(tokenized_datasets)
print(f"\nラベル: {label_list}")

=== データロード完了 ===
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1000
    })
})

ラベル: ['O', 'B-Chemical', 'I-Chemical', 'B-Disease', 'I-Disease']


In [3]:
# tokenizerを先にロード（insulinサンプル追加のため）
model_name = "dmis-lab/biobert-v1.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
print(f"トークナイザー: {model_name}")

トークナイザー: dmis-lab/biobert-v1.1


In [4]:
# insulinサンプルの追加（50サンプル）
insulin_samples = [
    # 基本的なパターン
    {"tokens": ["Patient", "takes", "insulin", "for", "diabetes", "mellitus"], "tags": [0, 0, 1, 0, 3, 4]},
    {"tokens": ["The", "patient", "was", "prescribed", "insulin", "for", "type", "2", "diabetes"], "tags": [0, 0, 0, 0, 1, 0, 0, 0, 3]},
    {"tokens": ["Insulin", "therapy", "is", "effective", "for", "diabetes", "management"], "tags": [1, 0, 0, 0, 0, 3, 0]},
    {"tokens": ["Treatment", "with", "insulin", "reduced", "blood", "sugar", "in", "diabetes"], "tags": [0, 0, 1, 0, 0, 0, 0, 3]},
    {"tokens": ["Diabetic", "patients", "often", "require", "insulin", "injections"], "tags": [3, 0, 0, 0, 1, 0]},
    
    # さまざまな文脈
    {"tokens": ["Daily", "insulin", "injections", "help", "control", "blood", "glucose"], "tags": [0, 1, 0, 0, 0, 0, 0]},
    {"tokens": ["The", "doctor", "recommended", "insulin", "for", "diabetes"], "tags": [0, 0, 0, 1, 0, 3]},
    {"tokens": ["Insulin", "is", "a", "hormone", "that", "regulates", "blood", "sugar"], "tags": [1, 0, 0, 0, 0, 0, 0, 0]},
    {"tokens": ["Patients", "with", "diabetes", "need", "regular", "insulin"], "tags": [0, 0, 3, 0, 0, 1]},
    {"tokens": ["Insulin", "deficiency", "causes", "high", "blood", "sugar"], "tags": [1, 0, 0, 0, 0, 0]},
    
    # 異なるパターン
    {"tokens": ["The", "nurse", "administered", "insulin", "to", "the", "patient"], "tags": [0, 0, 0, 1, 0, 0, 0]},
    {"tokens": ["Insulin", "resistance", "is", "common", "in", "type", "2", "diabetes"], "tags": [1, 0, 0, 0, 0, 0, 0, 3]},
    {"tokens": ["Long-acting", "insulin", "provides", "steady", "coverage"], "tags": [0, 1, 0, 0, 0]},
    {"tokens": ["Rapid-acting", "insulin", "is", "taken", "before", "meals"], "tags": [0, 1, 0, 0, 0, 0]},
    {"tokens": ["The", "patient", "uses", "an", "insulin", "pump"], "tags": [0, 0, 0, 0, 1, 0]},
    
    # 治療シーン
    {"tokens": ["Insulin", "treatment", "was", "started", "for", "diabetes"], "tags": [1, 0, 0, 0, 0, 3]},
    {"tokens": ["The", "dosage", "of", "insulin", "was", "increased"], "tags": [0, 0, 0, 1, 0, 0]},
    {"tokens": ["Insulin", "helps", "lower", "blood", "glucose", "levels"], "tags": [1, 0, 0, 0, 0, 0]},
    {"tokens": ["Multiple", "daily", "insulin", "injections", "are", "required"], "tags": [0, 0, 1, 0, 0, 0]},
    {"tokens": ["The", "patient", "responded", "well", "to", "insulin"], "tags": [0, 0, 0, 0, 0, 1]},
    
    # 医療文書風
    {"tokens": ["Clinical", "trial", "showed", "insulin", "efficacy"], "tags": [0, 0, 0, 1, 0]},
    {"tokens": ["Insulin", "therapy", "improved", "glycemic", "control"], "tags": [1, 0, 0, 0, 0]},
    {"tokens": ["The", "study", "compared", "different", "types", "of", "insulin"], "tags": [0, 0, 0, 0, 0, 0, 1]},
    {"tokens": ["Basal", "insulin", "provides", "background", "coverage"], "tags": [0, 1, 0, 0, 0]},
    {"tokens": ["Bolus", "insulin", "is", "given", "with", "meals"], "tags": [0, 1, 0, 0, 0, 0]},
    
    # さらなるバリエーション
    {"tokens": ["Insulin", "is", "essential", "for", "treating", "diabetes"], "tags": [1, 0, 0, 0, 0, 3]},
    {"tokens": ["The", "hospital", "uses", "human", "insulin"], "tags": [0, 0, 0, 0, 1]},
    {"tokens": ["Analogue", "insulin", "has", "faster", "onset"], "tags": [0, 1, 0, 0, 0]},
    {"tokens": ["Insulin", "storage", "requires", "refrigeration"], "tags": [1, 0, 0, 0]},
    {"tokens": ["The", "cost", "of", "insulin", "has", "increased"], "tags": [0, 0, 0, 1, 0, 0]},
    
    # 患者ケア
    {"tokens": ["Patient", "education", "includes", "insulin", "administration"], "tags": [0, 0, 0, 1, 0]},
    {"tokens": ["Proper", "insulin", "technique", "is", "important"], "tags": [0, 1, 0, 0, 0]},
    {"tokens": ["The", "clinic", "provides", "insulin", "training"], "tags": [0, 0, 0, 1, 0]},
    {"tokens": ["Insulin", "syringes", "are", "medical", "supplies"], "tags": [1, 0, 0, 0, 0]},
    {"tokens": ["Rotate", "injection", "sites", "for", "insulin"], "tags": [0, 0, 0, 0, 1]},
    
    # 症例
    {"tokens": ["Case", "report", "describes", "insulin", "use"], "tags": [0, 0, 0, 1, 0]},
    {"tokens": ["The", "patient", "was", "started", "on", "insulin"], "tags": [0, 0, 0, 0, 0, 1]},
    {"tokens": ["Insulin", "dose", "adjustment", "was", "necessary"], "tags": [1, 0, 0, 0, 0]},
    {"tokens": ["The", "patient", "monitors", "insulin", "levels"], "tags": [0, 0, 0, 1, 0]},
    {"tokens": ["Regular", "insulin", "acts", "quickly"], "tags": [0, 1, 0, 0]},
    
    # 追加サンプル
    {"tokens": ["NPH", "insulin", "is", "intermediate-acting"], "tags": [0, 1, 0, 0]},
    {"tokens": ["Glargine", "insulin", "lasts", "24", "hours"], "tags": [0, 1, 0, 0, 0]},
    {"tokens": ["The", "patient", "takes", "insulin", "twice", "daily"], "tags": [0, 0, 0, 1, 0, 0]},
    {"tokens": ["Insulin", "sensitivity", "varies", "among", "patients"], "tags": [1, 0, 0, 0, 0]},
    {"tokens": ["The", "protocol", "includes", "insulin", "therapy"], "tags": [0, 0, 0, 1, 0]},
    
    # 最後のサンプル
    {"tokens": ["Insulin", "is", "lifesaving", "for", "diabetes"], "tags": [1, 0, 0, 0, 3]},
    {"tokens": ["The", "patient", "was", "diagnosed", "with", "diabetes", "and", "started", "insulin"], "tags": [0, 0, 0, 0, 0, 3, 0, 0, 1]},
    {"tokens": ["Insulin", "helps", "manage", "blood", "sugar", "in", "diabetes"], "tags": [1, 0, 0, 0, 0, 0, 3]},
    {"tokens": ["The", "effectiveness", "of", "insulin", "was", "confirmed"], "tags": [0, 0, 0, 1, 0, 0]},
    {"tokens": ["Patients", "using", "insulin", "showed", "improvement"], "tags": [0, 0, 1, 0, 0]},
]

print(f"insulinサンプル数: {len(insulin_samples)}")
print("サンプル内容（最初の5個）:")
for i, sample in enumerate(insulin_samples[:5], 1):
    tokens = sample['tokens']
    tags = sample['tags']
    chemical_tokens = [tokens[j] for j, tag in enumerate(tags) if tag in [1, 2]]
    disease_tokens = [tokens[j] for j, tag in enumerate(tags) if tag in [3, 4]]
    print(f"  Sample {i}: Chemical={chemical_tokens}, Disease={disease_tokens}")
print(f"... （残り{len(insulin_samples)-5}サンプル）")

# トークナイズ
def tokenize_fn(examples):
    tokenized = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, tag in enumerate(examples["tags"]):
        word_ids = tokenized.word_ids(i)
        label_ids = []
        prev = None
        for wid in word_ids:
            if wid is None:
                label_ids.append(-100)
            elif wid != prev:
                label_ids.append(tag[wid])
            else:
                label_ids.append(-100)
            prev = wid
        labels.append(label_ids)
    tokenized["labels"] = labels
    return tokenized

# insulinサンプルをトークナイズ
from datasets import Dataset
insulin_ds = Dataset.from_list(insulin_samples)
insulin_tokenized = insulin_ds.map(tokenize_fn, batched=True, remove_columns=['tokens', 'tags'])

# 既存データと結合
from datasets import concatenate_datasets
tokenized_datasets["train"] = concatenate_datasets([tokenized_datasets["train"], insulin_tokenized])

print(f"\n追加後のトレーニングデータ: {len(tokenized_datasets['train'])} サンプル")
print("(元の1000サンプル + insulinの50サンプル = 1050サンプル)")

insulinサンプル数: 50
サンプル内容（最初の5個）:
  Sample 1: Chemical=['insulin'], Disease=['diabetes', 'mellitus']
  Sample 2: Chemical=['insulin'], Disease=['diabetes']
  Sample 3: Chemical=['Insulin'], Disease=['diabetes']
  Sample 4: Chemical=['insulin'], Disease=['diabetes']
  Sample 5: Chemical=['insulin'], Disease=['Diabetic']
... （残り45サンプル）


Map:   0%|          | 0/50 [00:00<?, ? examples/s]


追加後のトレーニングデータ: 1050 サンプル
(元の1000サンプル + insulinの50サンプル = 1050サンプル)


In [5]:
# モデルロード
# tokenizerは上で既にロード済み
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"モデル: {model_name}")
print(f"パラメータ数: {model.num_parameters():,}")
print(f"ラベル数: {num_labels}")

Some weights of BertForTokenClassification were not initialized from the model checkpoint at dmis-lab/biobert-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


モデル: dmis-lab/biobert-v1.1
パラメータ数: 107,723,525
ラベル数: 5


In [6]:
# 評価関数
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    
    # -100を除外
    true_preds = []
    true_labels = []
    for pred, label in zip(predictions, labels):
        for p, l in zip(pred, label):
            if l != -100:
                true_preds.append(label_list[p])
                true_labels.append(label_list[l])
    
    # F1計算（micro平均）
    from sklearn.metrics import f1_score
    f1 = f1_score(true_labels, true_preds, average='micro')
    
    return {'f1': f1}

print("評価関数定義完了")

評価関数定義完了


In [7]:
# トレーニング設定
training_args = TrainingArguments(
    output_dir="../results/checkpoint",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="../results/logs",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True
)

data_collator = DataCollatorForTokenClassification(tokenizer)

print("=== トレーニング設定 ===")
print(f"エポック数: {training_args.num_train_epochs}")
print(f"バッチサイズ: {training_args.per_device_train_batch_size}")
print(f"学習率: {training_args.learning_rate}")

=== トレーニング設定 ===
エポック数: 3
バッチサイズ: 16
学習率: 2e-05


C:\Users\hello\Desktop\project\PyTorch_Proj\.conda\env\Lib\site-packages\transformers\training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [8]:
# Trainer初期化
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer初期化完了")

C:\Users\hello\AppData\Local\Temp\ipykernel_22368\1682483468.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Trainer初期化完了


## トレーニング

**注意**: GPUで約10-15分かかります

In [9]:
print("トレーニング開始...")
trainer.train()

print("\nトレーニング完了!")

トレーニング開始...


Epoch,Training Loss,Validation Loss,F1
1,No log,0.116291,0.963607
2,No log,0.087866,0.971919
3,No log,0.082750,0.973260



トレーニング完了!


In [10]:
# 評価
metrics = trainer.evaluate()

print("=== 最終評価 ===")
print(f"F1スコア: {metrics['eval_f1']:.4f}")

# モデル保存
final_path = "../results/biobert-ner-bc5cdr"
trainer.save_model(final_path)
tokenizer.save_pretrained(final_path)

print(f"\nモデル保存完了: {final_path}")
print("\n次: 03_evaluation.ipynb")

=== 最終評価 ===
F1スコア: 0.9733

モデル保存完了: ../results/biobert-ner-bc5cdr

次: 03_evaluation.ipynb
